In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import unyt as u

import dev
import richio


In [ ]:
DATADIR = Path("/home/hey4/rich_tde/data/processed/epsilondiss-t")

configs = {
    "1e4": {
        "Rstar": 0.47 * richio.units.lscale,
        "Mstar": 0.5 * richio.units.mscale,
        "Mbh": 1e4 * richio.units.mscale,
    },
    "1e5": {
        "Rstar": 0.47 * richio.units.lscale,
        "Mstar": 0.5 * richio.units.mscale,
        "Mbh": 1e5 * richio.units.mscale,
    },
    "1e6": {
        "Rstar": 1 * richio.units.lscale,
        "Mstar": 1 * richio.units.mscale,
        "Mbh": 1e6 * richio.units.mscale,
    },
}

for mode, cfg in configs.items():
    Rstar, Mstar, Mbh = cfg["Rstar"], cfg["Mstar"], cfg["Mbh"]
    r_p = Rstar * (Mbh / Mstar) ** (1 / 3)
    cfg["tmin"] = (
        np.pi
        / np.sqrt(2)
        * (Rstar**3 / u.G / Mstar) ** (1 / 2)
        * (Mbh / Mstar) ** (1 / 2)
    )
    cfg["Delta"] = u.G * Mbh / (4 * r_p)
    cfg["DATAFILE"] = DATADIR / f"epsilondiss-t-{mode}-final.txt"


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

reg = richio.units.registry
regions = [
    "Pericenter side $(x>0)$",
    "Outgoing stream $(x<0, y<0)$",
    "Incoming stream $(x<0, y>0)$",
]
region_styles = ["-", ":", "--", "-."]
mode_colors = {"1e4": "C0", "1e5": "C1", "1e6": "C2"}
mode_labels = {
    "1e4": r"$10^4\,M_\odot$",
    "1e5": r"$10^5\,M_\odot$",
    "1e6": r"$10^6\,M_\odot$",
}

def load_timeseries(path):
    raw = np.atleast_2d(np.loadtxt(path, delimiter="\t"))
    if raw.shape[1] != 7:
        raise ValueError(f"{path} has {raw.shape[1]} columns; expected 7")
    if not np.isfinite(raw[:, :3]).all():
        raise ValueError(f"{path} has invalid snapshot/time columns")
    if np.isinf(raw[:, 3:]).any():
        raise ValueError(f"{path} has infinite specific-dissipation values")
    return raw[np.argsort(raw[:, 1])].T

def finite_ratio(numerator, denominator):
    numerator = numerator.to_value()
    denominator = denominator.to_value()
    ratio = np.full(numerator.shape, np.nan)
    valid = (
        np.isfinite(numerator)
        & np.isfinite(denominator)
        & (denominator != 0)
    )
    np.divide(numerator, denominator, out=ratio, where=valid)
    return ratio

def integrate_finite_intervals(rate, time):
    increments = 0.5 * (rate[1:] + rate[:-1]) * np.diff(time)
    increments = increments.to_value()
    valid = np.isfinite(increments)
    integrated = np.cumsum(np.where(valid, increments, 0.0))
    integrated[~valid] = np.nan
    return integrated

plt.figure(figsize=(6, 4), dpi=300)
for mode, cfg in configs.items():
    raw = load_timeseries(cfg["DATAFILE"])
    raw = raw[:, raw[2] >= 0.1]
    if mode == "1e6":
        raw = raw[:, raw[2] >= 0.7]
    t_in_tmins = raw[2]
    epsilondiss = u.unyt_array(
        raw[3:6], "code_length**2/code_time**3", registry=reg
    )
    epsilondiss_Deltas = epsilondiss / cfg["Delta"] * cfg["tmin"]

    for row, style in zip(epsilondiss_Deltas, region_styles):
        plt.plot(
            t_in_tmins, row, color=mode_colors[mode],
            linestyle=style, linewidth=1.5,
        )

plt.xlabel(r"$t[t_\mathrm{fb}]$")
plt.ylabel(r"$\dot{\epsilon}_\mathrm{diss}/\Delta\epsilon_c\,[1/t_\mathrm{fb}]$")
plt.yscale("log")
plt.ylim(1e-8, 1)

region_handles = [
    Line2D([], [], color="k", linestyle=s, label=r)
    for s, r in zip(region_styles, regions)
]
mode_handles = [
    Line2D([], [], color=mode_colors[m], label=mode_labels[m]) for m in configs
]
region_legend = plt.legend(handles=region_handles, loc="lower right", fontsize=8)
plt.gca().add_artist(region_legend)
plt.legend(
    handles=mode_handles, title=r"$M_\mathrm{BH}$",
    loc="upper right", fontsize=8,
)
plt.show()


In [ ]:
ratio_styles = ["-", "--"]
ratio_labels = ["Nozzle / incoming stream", "Nozzle / outgoing stream"]

plt.figure(figsize=(6, 4), dpi=300)
for mode, cfg in configs.items():
    raw = load_timeseries(cfg["DATAFILE"])
    raw = raw[:, raw[2] >= 0.1]
    t_in_tmins = raw[2]
    epsilondiss = u.unyt_array(
        raw[3:7], "code_length**2/code_time**3", registry=reg
    )
    epsilondiss1, epsilondiss2, epsilondiss3, _ = epsilondiss

    plt.plot(
        t_in_tmins, finite_ratio(epsilondiss1, epsilondiss3),
        color=mode_colors[mode], linestyle=ratio_styles[0], linewidth=1.5,
    )
    plt.plot(
        t_in_tmins, finite_ratio(epsilondiss1, epsilondiss2),
        color=mode_colors[mode], linestyle=ratio_styles[1], linewidth=1.5,
    )

plt.xlabel(r"$t[t_\mathrm{fb}]$")
plt.ylabel("Ratio")
plt.yscale("log")
plt.ylim(1e0, 1e3)

ratio_handles = [
    Line2D([], [], color="k", linestyle=s, label=l)
    for s, l in zip(ratio_styles, ratio_labels)
]
mode_handles = [
    Line2D([], [], color=mode_colors[m], label=mode_labels[m]) for m in configs
]
ratio_legend = plt.legend(handles=ratio_handles, loc="lower right", fontsize=8)
plt.gca().add_artist(ratio_legend)
plt.legend(
    handles=mode_handles, title=r"$M_\mathrm{BH}$",
    loc="upper right", fontsize=8,
)
plt.show()


In [ ]:
plt.figure(figsize=(6, 4), dpi=300)
for mode, cfg in configs.items():
    raw = load_timeseries(cfg["DATAFILE"])
    raw = raw[:, raw[2] >= 0.1]
    ts = u.unyt_array(raw[1], "code_time", registry=reg)
    t_in_tmins = raw[2]
    epsilondiss = u.unyt_array(
        raw[3:6], "code_length**2/code_time**3", registry=reg
    )
    epsilondiss_Deltas = epsilondiss / cfg["Delta"]

    for row, style in zip(epsilondiss_Deltas, region_styles):
        integrated = integrate_finite_intervals(row, ts)
        plt.plot(
            t_in_tmins[1:], integrated, color=mode_colors[mode],
            linestyle=style, linewidth=1.5,
        )

plt.xlabel(r"$t[t_\mathrm{fb}]$")
plt.ylabel(r"$\epsilon_\mathrm{diss}/\Delta\epsilon_c$")
plt.yscale("log")
plt.ylim(1e-6, 1)

region_handles = [
    Line2D([], [], color="k", linestyle=s, label=r)
    for s, r in zip(region_styles, regions)
]
mode_handles = [
    Line2D([], [], color=mode_colors[m], label=mode_labels[m]) for m in configs
]
region_legend = plt.legend(handles=region_handles, loc="lower right", fontsize=8)
plt.gca().add_artist(region_legend)
plt.legend(
    handles=mode_handles, title=r"$M_\mathrm{BH}$",
    loc="upper right", fontsize=8,
)
plt.show()
